# 🧠 Stochastic Gradient Descent (SGD)

Welcome to the hands-on explanation notebook for **Stochastic Gradient Descent (SGD)**! In this notebook, we will:
1. Compare Batch Gradient Descent vs. Stochastic Gradient Descent conceptually and mathematically.
2. Implement both Batch GD and SGD from scratch in Python/NumPy.
3. Track and compare their weight update trajectories on a 2D loss contour map.
4. Plot loss curves to visualize the smooth convergence of Batch GD vs. the noisy fluctuations of SGD.
5. Explain the role of stochastic noise in escaping local minima and its hardware efficiency limitations (leading to Mini-Batch GD used in YOLO).

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

## 1. Simulating Data for Linear Regression

We generate 100 points following the linear relationship:
$$y = 3x + 1 + \text{noise}$$

In [ ]:
n_samples = 100
X = np.random.rand(n_samples, 1)
y = 3 * X + 1 + np.random.normal(0, 0.15, (n_samples, 1))

plt.figure(figsize=(8, 5))
plt.scatter(X, y, color='blue', edgecolor='k', s=40, label='Data Points')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Synthetic Linear Dataset')
plt.grid(True, linestyle='--', alpha=0.3)
plt.legend()
plt.show()

## 2. Implementing Batch GD and SGD from Scratch

Let's write functions to track the path of weights $w$ and bias $b$ as they optimize.
-   In **Batch GD**, we perform 1 update per epoch using the average gradient of all 100 samples.
-   In **SGD**, we perform 100 updates per epoch (updating after every individual sample, shuffled randomly).

In [ ]:
def run_batch_gd(X, y, lr=0.1, epochs=50):
    w, b = 0.0, 0.0
    m = len(X)
    history = [[w, b]]
    
    for _ in range(epochs):
        preds = w * X + b
        error = preds - y
        dw = (1 / m) * np.sum(error * X)
        db = (1 / m) * np.sum(error)
        w -= lr * dw
        b -= lr * db
        history.append([w, b])
        
    return np.array(history)

def run_sgd(X, y, lr=0.1, epochs=5):
    w, b = 0.0, 0.0
    m = len(X)
    history = [[w, b]]
    
    for _ in range(epochs):
        indices = np.random.permutation(m)
        for idx in indices:
            xi = X[idx]
            yi = y[idx]
            pred = w * xi + b
            error = pred - yi
            dw = error * xi
            db = error
            w -= lr * dw
            b -= lr * db
            history.append([float(w), float(b)])
            
    return np.array(history)

path_bgd = run_batch_gd(X, y, lr=0.1, epochs=50)
path_sgd = run_sgd(X, y, lr=0.02, epochs=5)

## 3. Visualizing Trajectories on the 2D Loss Contour Map

Let's compute the total Loss Landscape and plot the optimization paths.

In [ ]:
# Compute loss grid
w_vals = np.linspace(-0.5, 4.5, 100)
b_vals = np.linspace(-0.5, 2.5, 100)
W, B = np.meshgrid(w_vals, b_vals)

Z = np.zeros_like(W)
for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        w_tmp, b_tmp = W[i, j], B[i, j]
        Z[i, j] = (1 / (2 * n_samples)) * np.sum((w_tmp * X + b_tmp - y) ** 2)

# Plot contours and trajectories
plt.figure(figsize=(10, 8))
contours = plt.contour(W, B, Z, levels=25, cmap='viridis')
plt.clabel(contours, inline=1, fontsize=8)

plt.plot(path_bgd[:, 0], path_bgd[:, 1], color='red', marker='o', linewidth=2.5, label='Batch GD (Smooth path)')
plt.plot(path_sgd[:, 0], path_sgd[:, 1], color='orange', alpha=0.7, linewidth=1.5, label='SGD (Noisy/Jagged path)')

plt.scatter(3.0, 1.0, color='blue', s=120, marker='*', zorder=5, label='Target Minimum')
plt.xlabel('Weight (w)')
plt.ylabel('Bias (b)')
plt.title('Optimization Paths: Batch GD vs. Stochastic GD')
plt.legend()
plt.show()

Look at the plot!
-   **Batch GD (Red):** Moves in a perfectly straight, perpendicular line down the gradient, converging slowly but cleanly.
-   **SGD (Orange):** Takes a highly chaotic, zigzag path. The random noise bounces the parameters around, but overall it converges toward the minimum much faster in terms of data processed!

## 💡 Connection to YOLO and hardware efficiency
Why does YOLO use **Mini-Batch GD** instead of pure SGD (batch size = 1)?
1.  **Vectorization & Parallelism:** Processors (especially GPUs) are designed to perform matrix multiplications. Multiplying a weight matrix by a batch of 16 or 32 images takes almost the same time as multiplying it by 1 image because of parallel execution.
2.  **Gradient Stability:** Pure SGD has too much noise, preventing it from converging to high accuracy. Mini-batching averages gradients over 16-64 images, stabilizing updates while maintaining high processing speeds.